In [ ]:
import os
import shutil
import zipfile
import pandas as pd

In [ ]:
from google.colab import files

In [ ]:
# ==============================================================================
# 1. SETUP & EXTRACT DATA
# ==============================================================================

In [ ]:
print("--- [1/5] Memuat dan Mengandangkan File ZIP ---")
uploaded = files.upload()
zip_file = list(uploaded.keys())[0]

--- [1/5] Memuat dan Mengandangkan File ZIP ---


Saving archive (1).zip to archive (1).zip


In [ ]:
extract_dir = "olist_raw_data"
with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"File berhasil diekstrak ke folder: '{extract_dir}'")

File berhasil diekstrak ke folder: 'olist_raw_data'


In [ ]:
# ==============================================================================
# 2. SAMPLING & SUBSETTING DATA
# ==============================================================================

In [ ]:
print("\n--- [2/5] Melakukan Subsetting Data (10,000 Orders) ---")

# a. Sampling Orders
orders = pd.read_csv(os.path.join(extract_dir, "olist_orders_dataset.csv"))
selected_orders = orders.sample(n=10000, random_state=42).copy()

# Datetime & Validation Meta
selected_orders["order_purchase_timestamp"] = pd.to_datetime(
    selected_orders["order_purchase_timestamp"]
)
selected_orders["year"] = selected_orders["order_purchase_timestamp"].dt.year

# Extract Primary Keys untuk Filtering
order_ids = set(selected_orders["order_id"])
customer_ids = set(selected_orders["customer_id"])

# b. Load & Filter Relational Tables
order_items = pd.read_csv(
    os.path.join(extract_dir, "olist_order_items_dataset.csv")
)
order_items_subset = order_items[
    order_items["order_id"].isin(order_ids)
].copy()

product_ids = set(order_items_subset["product_id"])
seller_ids = set(order_items_subset["seller_id"])

customers = pd.read_csv(
    os.path.join(extract_dir, "olist_customers_dataset.csv")
)
customers_subset = customers[
    customers["customer_id"].isin(customer_ids)
].copy()

payments = pd.read_csv(
    os.path.join(extract_dir, "olist_order_payments_dataset.csv")
)
payments_subset = payments[payments["order_id"].isin(order_ids)].copy()

reviews = pd.read_csv(
    os.path.join(extract_dir, "olist_order_reviews_dataset.csv")
)
reviews_subset = reviews[reviews["order_id"].isin(order_ids)].copy()

products = pd.read_csv(os.path.join(extract_dir, "olist_products_dataset.csv"))
products_subset = products[products["product_id"].isin(product_ids)].copy()

sellers = pd.read_csv(os.path.join(extract_dir, "olist_sellers_dataset.csv"))
sellers_subset = sellers[sellers["seller_id"].isin(seller_ids)].copy()

translation = pd.read_csv(
    os.path.join(extract_dir, "product_category_name_translation.csv")
)

print("Proses filtering subset selesai secara komprehensif.")


--- [2/5] Melakukan Subsetting Data (10,000 Orders) ---
Proses filtering subset selesai secara komprehensif.


In [ ]:
# ==============================================================================
# 3. VALIDASI INTEGRITAS RELASI DATA
# ==============================================================================

In [ ]:
print("\n--- [3/5] Memeriksa Validasi Relasi Data ---")

validations = {
    "Order Items -> Orders": ~order_items_subset["order_id"].isin(
        selected_orders["order_id"]
    ),
    "Order Items -> Products": ~order_items_subset["product_id"].isin(
        products_subset["product_id"]
    ),
    "Order Items -> Sellers": ~order_items_subset["seller_id"].isin(
        sellers_subset["seller_id"]
    ),
    "Payments -> Orders": ~payments_subset["order_id"].isin(
        selected_orders["order_id"]
    ),
    "Reviews -> Orders": ~reviews_subset["order_id"].isin(
        selected_orders["order_id"]
    ),
    "Orders -> Customers": ~selected_orders["customer_id"].isin(
        customers_subset["customer_id"]
    ),
}

all_valid = True
for rel_name, mask in validations.items():
    missing_count = mask.sum()
    print(f"• {rel_name}: {missing_count} orphan records")
    if missing_count > 0:
        all_valid = False

if all_valid:
    print("STATUS INTEGRITAS: PERFECT (Seluruh kunci asing selaras).")


--- [3/5] Memeriksa Validasi Relasi Data ---
• Order Items -> Orders: 0 orphan records
• Order Items -> Products: 0 orphan records
• Order Items -> Sellers: 0 orphan records
• Payments -> Orders: 0 orphan records
• Reviews -> Orders: 0 orphan records
• Orders -> Customers: 0 orphan records
STATUS INTEGRITAS: PERFECT (Seluruh kunci asing selaras).


In [ ]:
# ==============================================================================
# 4. EXPORT FILE CSV & ZIP
# ==============================================================================

In [ ]:
print("\n--- [4/5] Meng-export File Subset dan Membuat Arsip ZIP ---")
output_dir = "olist_subset_10000"
os.makedirs(output_dir, exist_ok=True)

export_mapping = {
    "orders.csv": selected_orders,
    "customers.csv": customers_subset,
    "order_items.csv": order_items_subset,
    "payments.csv": payments_subset,
    "reviews.csv": reviews_subset,
    "products.csv": products_subset,
    "sellers.csv": sellers_subset,
    "category_translation.csv": translation,
}

for filename, df in export_mapping.items():
    df.to_csv(os.path.join(output_dir, filename), index=False)

zip_filename = "olist_subset_10000"
shutil.make_archive(zip_filename, "zip", output_dir)
print(f"Arsip '{zip_filename}.zip' berhasil dibuat.")


--- [4/5] Meng-export File Subset dan Membuat Arsip ZIP ---
Arsip 'olist_subset_10000.zip' berhasil dibuat.


In [ ]:
# ==============================================================================
# 5. DOWNLOAD AUTOMATION
# ==============================================================================

In [ ]:
print("\n--- [5/5] Mengunduh File ZIP ---")
files.download(f"{zip_filename}.zip")


--- [5/5] Mengunduh File ZIP ---


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>